# Act 5 — Querying MLflow Programmatically

> *The UI shows you what happened. The API lets you act on it.*

**Goal:** Pull MLflow experiments, runs, eval metrics, and traces into Python for custom analysis —
dashboards, alerts, regression detection, audit logs.

**Run notebooks 01–04 first** — this notebook queries the data they produced.

| API | What it queries |
|---|---|
| `mlflow.search_experiments()` | List all experiments |
| `mlflow.search_runs()` | Eval run metrics (filter, compare, rank) |
| `mlflow.search_traces()` | Traces (latency, status, tags) |
| `MlflowClient` | Low-level: artifacts, run details, pagination |

In [1]:
import mlflow
import pandas as pd
from mlflow import MlflowClient

mlflow.set_tracking_uri("http://127.0.0.1:5000")
client = MlflowClient()

pd.set_option("display.max_colwidth", 50)
pd.set_option("display.max_columns", 20)

print("Connected to MLflow at http://127.0.0.1:5000")

Connected to MLflow at http://127.0.0.1:5000


---
## Query 1 — List experiments

In [2]:
experiments = mlflow.search_experiments()

df_exps = pd.DataFrame([
    {"name": e.name, "experiment_id": e.experiment_id, "lifecycle": e.lifecycle_stage}
    for e in experiments
    if e.name != "Default"
])

print(f"{len(df_exps)} experiments found:")
df_exps

5 experiments found:


,name,experiment_id,lifecycle
0,Session4 / Act 4 - RAG & Agent Eval,6,active
1,Session4 / Act 3 - Guardrails,5,active
2,Session4 / Act 2 - Evaluation-Metrics,4,active
3,Session4 / Act 1 - Tracing,2,active
4,ZomatoBot,1,active


---
## Query 2 — Search eval runs

In [3]:
runs = mlflow.search_runs(
    experiment_names=["Session4 / Act 2 - Evaluation Metrics"],
    order_by=["start_time DESC"],
)

metric_cols = [c for c in runs.columns if c.startswith("metrics.")]
display_cols = ["run_id", "start_time", "status"] + metric_cols[:6]

print(f"{len(runs)} runs found. Metrics columns: {metric_cols}")
runs[display_cols].head(5)

0 runs found. Metrics columns: []


,run_id,start_time,status


---
## Query 3 — Filter runs by metric threshold

In [4]:
# Find runs where politeness pass rate >= 0.8
good_runs = mlflow.search_runs(
    experiment_names=["Session4 / Act 2 - Evaluation Metrics"],
    filter_string="metrics.`is_polite/mean` >= 0.8",
    order_by=["metrics.`is_polite/mean` DESC"],
)

print(f"Runs with is_polite ≥ 0.8: {len(good_runs)}")
if not good_runs.empty:
    print(good_runs[["run_id", "metrics.is_polite/mean"]].to_string(index=False))

Runs with is_polite ≥ 0.8: 0


---
## Query 4 — Compare two runs side-by-side

In [5]:
all_runs = mlflow.search_runs(
    experiment_names=["Session4 / Act 2 - Evaluation Metrics"],
    order_by=["start_time DESC"],
)

metric_cols = [c for c in all_runs.columns if c.startswith("metrics.")]

if len(all_runs) >= 2:
    run_a = all_runs.iloc[0]
    run_b = all_runs.iloc[1]
    comparison = pd.DataFrame({
        "metric": [c.replace("metrics.", "") for c in metric_cols],
        "run_A (latest)": [run_a[c] for c in metric_cols],
        "run_B (prev)":   [run_b[c] for c in metric_cols],
    }).dropna()
    comparison["delta"] = comparison["run_A (latest)"] - comparison["run_B (prev)"]
    print(comparison.to_string(index=False))
else:
    print("Need at least 2 runs to compare — re-run notebook 02 to generate a second run.")

Need at least 2 runs to compare — re-run notebook 02 to generate a second run.


---
## Query 5 — Search traces by status

In [6]:
# Get the experiment ID for Act 1 tracing
tracing_exp = mlflow.get_experiment_by_name("Session4 / Act 1 - Tracing")

if tracing_exp:
    traces = mlflow.search_traces(
        experiment_ids=[tracing_exp.experiment_id],
        filter_string="status = 'OK'",
        order_by=["timestamp_ms DESC"],
        return_type="pandas",
    )
    print(f"{len(traces)} successful traces in Act 1")
    if not traces.empty:
        display_cols = [c for c in ["trace_id", "state", "execution_duration", "request"] if c in traces.columns]
        print(traces[display_cols].head(5).to_string())
else:
    print("Run notebook 01 first to generate tracing data.")

3 successful traces in Act 1
                              trace_id state  execution_duration                                                                                                                                                                                                                                                                                                                request
0  tr-a0a12077b1865471d602fd3a8a1f6d01    OK                2277                                                                                                                                                                                                                                               {'query': 'Cheap veg thali under Rs.100 in Pune?', 'user_id': 'user-42'}
1  tr-f12827213e7ff4c69736639a26f7d62f    OK                1596  {'model': 'llama3.2:1b', 'messages': [{'role': 'system', 'content': 'You are ZomatoBot, a helpful restaurant assistant for Indian cities. Answer in 2-3 s

/var/folders/6v/f9s5l9sx2n71w2lry5q1yhw00000gn/T/ipykernel_82230/1402154575.py:5: FutureWarning: Parameter 'experiment_ids' is deprecated. Please use 'locations' instead.
  traces = mlflow.search_traces(


---
## Query 6 — Filter traces by tag (user-level analysis)

In [7]:
# Find all traces tagged with a specific user (set in Act 1 Part B)
if tracing_exp:
    user_traces = mlflow.search_traces(
        experiment_ids=[tracing_exp.experiment_id],
        filter_string="tags.`user_id` = 'user-42'",
        return_type="pandas",
    )
    print(f"Traces for user-42: {len(user_traces)}")
    if user_traces.empty:
        print("  (No tagged traces found — the tag is set in Act 1 Part B)")
    else:
        print(user_traces[["trace_id", "execution_duration"]].to_string())
else:
    print("Run notebook 01 first.")

Traces for user-42: 1
                              trace_id  execution_duration
0  tr-a0a12077b1865471d602fd3a8a1f6d01                2277


/var/folders/6v/f9s5l9sx2n71w2lry5q1yhw00000gn/T/ipykernel_82230/2730330874.py:3: FutureWarning: Parameter 'experiment_ids' is deprecated. Please use 'locations' instead.
  user_traces = mlflow.search_traces(


---
## Query 7 — Latency analysis (find slow traces)

In [8]:
# Get all Session4 experiment IDs
all_exps = mlflow.search_experiments()
session4_ids = [
    e.experiment_id for e in all_exps
    if e.name.startswith("Session4")
]

if session4_ids:
    all_traces = mlflow.search_traces(
        experiment_ids=session4_ids,
        filter_string="status = 'OK'",
        order_by=["execution_time_ms DESC"],
        return_type="pandas",
        max_results=50,
    )
    if not all_traces.empty and "execution_duration" in all_traces.columns:
        all_traces["latency_s"] = pd.to_numeric(
            all_traces["execution_duration"], errors="coerce"
        ) / 1000
        print(f"All traces — latency summary (seconds):")
        print(all_traces["latency_s"].describe().round(2))
        print("\nTop 5 slowest traces:")
        slowest_cols = [c for c in ["trace_id", "latency_s"] if c in all_traces.columns]
        print(all_traces.nlargest(5, "latency_s")[slowest_cols].to_string(index=False))
    else:
        print(f"Found {len(all_traces)} traces but execution_duration column not available.")
        print("Available columns:", list(all_traces.columns))
else:
    print("No Session4 experiments found — run notebooks 01–04 first.")

/var/folders/6v/f9s5l9sx2n71w2lry5q1yhw00000gn/T/ipykernel_82230/3334580305.py:9: FutureWarning: Parameter 'experiment_ids' is deprecated. Please use 'locations' instead.
  all_traces = mlflow.search_traces(


All traces — latency summary (seconds):
count    34.00
mean      2.31
std       2.44
min       0.01
25%       0.64
50%       1.54
75%       2.76
max       9.83
Name: latency_s, dtype: float64

Top 5 slowest traces:
                           trace_id  latency_s
tr-357aae39e902ef6c407183188475dc1a      9.832
tr-11944945272bbca3a2bb863ca0e7ac16      8.377
tr-c141497a6eaca1d6d102e15dd2c430e6      7.557
tr-6756f3eab73f74c6d2da2719988b79f7      6.389
tr-1eb056b00aa7f5ad8035c0fb9ce792a9      4.190


---
## Query 8 — Custom metric dashboard

Pull all eval metrics into a DataFrame for trend analysis across runs.

In [9]:
# Aggregate all metrics across all Session4 eval experiments
all_metrics = []

session4_exp_names = [
    "Session4 / Act 2 - Evaluation Metrics",
    "Session4 / Act 3 - Guardrails",
    "Session4 / Act 4 - RAG & Agent Eval",
]

for exp_name in session4_exp_names:
    try:
        runs = mlflow.search_runs(
            experiment_names=[exp_name],
            order_by=["start_time DESC"],
            max_results=1,  # latest run only
        )
        if runs.empty:
            continue
        latest = runs.iloc[0]
        for col in latest.index:
            if col.startswith("metrics."):
                all_metrics.append({
                    "experiment": exp_name.split(" / ")[1],
                    "metric": col.replace("metrics.", ""),
                    "value": latest[col],
                })
    except Exception as e:
        print(f"  Skipped {exp_name}: {e}")

if all_metrics:
    dashboard = pd.DataFrame(all_metrics).dropna(subset=["value"])
    dashboard["value"] = dashboard["value"].round(2)
    print("=== Session 4 — Metric Dashboard ===")
    print(dashboard.to_string(index=False))
else:
    print("No metrics found — run notebooks 02–04 first.")

=== Session 4 — Metric Dashboard ===
              experiment                   metric  value
      Act 3 - Guardrails         not_blocked/mean   0.60
      Act 3 - Guardrails  response_length_ok/mean   1.00
Act 4 - RAG & Agent Eval correct_tool_called/mean   0.67
Act 4 - RAG & Agent Eval      task_completed/mean   0.50


---
## Summary — UI vs. API

| Task | Use UI | Use API |
|---|---|---|
| Explore a single trace | ✅ | — |
| Compare 2 runs visually | ✅ | — |
| Automated regression alerts | — | ✅ |
| Custom dashboards | — | ✅ |
| Filter traces by latency / tag | — | ✅ |
| Nightly eval in CI | — | ✅ |
| Audit log export | — | ✅ |

### Quick reference
```python
# Experiments
mlflow.search_experiments()

# Runs (with metric filter)
mlflow.search_runs(
    experiment_names=["..."],
    filter_string="metrics.`is_polite/mean` >= 0.8",
    order_by=["start_time DESC"],
)

# Traces (with tag / latency filter)
mlflow.search_traces(
    experiment_ids=["1", "2"],
    filter_string="status = 'OK' AND tags.`user_id` = 'user-42'",
    order_by=["execution_time_ms DESC"],
    return_type="dataframe",
)
```